In [53]:
import os

notebook_path = os.getcwd()

relative_path = os.path.join(notebook_path, "..", "..")
base_path = os.path.abspath(relative_path)
print("Base Path:", base_path)

#  D:/tierra/outputs/unfiltered/harmonized/mexico/Mexico_standardized_orgc.csv
input_file = os.path.join(base_path, "outputs", "unfiltered", "harmonized", "mexico", "Mexico_standardized_orgc.csv")
print("Input file path:", input_file)

existing_dataset = os.path.join(base_path, "datasets", "Mexico_standardized_cfvo_orgc.csv")
print("Existing dataset path:", existing_dataset)

Base Path: d:\tierra
Input file path: d:\tierra\outputs\unfiltered\harmonized\mexico\Mexico_standardized_orgc.csv
Existing dataset path: d:\tierra\datasets\Mexico_standardized_cfvo_orgc.csv


In [60]:
import pandas as pd

# Read both CSV files
df_existing = pd.read_csv(existing_dataset)
print(df_existing.columns)
df_input = pd.read_csv(input_file)
print(df_input.columns)

Index(['profile_id', 'depth_category', 'date', 'longitude', 'latitude', 'clay',
       'elcosp', 'phaq', 'sand', 'silt', 'orgc', 'soil_type', 'slopemean',
       'bedrock', 'cfr', 'temperature', 'precipitation', 'ecoregion_type',
       'zone_number', 'zone_name', 'landcover'],
      dtype='object')
Index(['profile_id', 'depth_category', 'date', 'longitude', 'latitude', 'clay',
       'phaq', 'sand', 'silt', 'orgc'],
      dtype='object')


In [55]:
# drop
df_existing.drop(columns=['landcover', 'zone_number'], inplace=True, errors='ignore')

# rename columns
df_existing.rename(columns={
    'ecoregion_type': 'ecoregion',
    'zone_name': 'landcover'
}, inplace=True)

In [56]:
# Merge the datasets based on profile_id
# Only selecting landcover, precipitation, ecoregion, temperature columns from existing dataset
columns_to_merge = ['profile_id', 'slopemean', 'bedrock', 'cfr', 'temperature', 'landcover',
                     'precipitation', 'ecoregion']
merged_df = pd.merge(
    df_input,
    df_existing[columns_to_merge],
    on='profile_id',
    how='left'
)

In [57]:
# pre-process
pd.set_option('display.max_columns', None)  # Show all columns in the output
print("Merged DataFrame shape:", merged_df.shape)
merged_df.head()

Merged DataFrame shape: (11026, 17)


,profile_id,depth_category,date,longitude,latitude,clay,phaq,sand,silt,orgc,slopemean,bedrock,cfr,temperature,landcover,precipitation,ecoregion
0,1147761,0_30,1991-10-29,-102.716667,20.866667,70.900000,6.100,7.500000,21.600000,1.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1149108,0_30,1986-10-12,-96.732749,15.775138,4.000000,5.500,66.000000,30.000000,10.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1149115,0_30,2004-5-27,-92.924872,15.375747,12.576397,5.882,42.139379,45.284223,11.860177,0.153699,rocas_metamorficas,0.001939,16.675776,Tropical húmeda,147.141876,Terrestre
3,1149115,0_30,2004-5-27,-92.924872,15.375747,12.576397,5.882,42.139379,45.284223,11.860177,0.153699,rocas_metamorficas,0.011414,16.675776,Tropical húmeda,147.141876,Terrestre
4,1149115,0_30,2004-5-27,-92.924872,15.375747,12.576397,5.882,42.139379,45.284223,11.860177,0.153699,rocas_metamorficas,0.000931,16.675776,Tropical húmeda,147.141876,Terrestre


In [58]:
# drop duplicates
merged_df.drop_duplicates(subset=['profile_id'], inplace=True)
print("After dropping duplicates, DataFrame shape:", merged_df.shape)

# drop rows with missing values
merged_df.dropna(inplace=True)
print("After dropping rows with missing values, DataFrame shape:", merged_df.shape)

merged_df.head()

After dropping duplicates, DataFrame shape: (4336, 17)
After dropping rows with missing values, DataFrame shape: (3308, 17)


,profile_id,depth_category,date,longitude,latitude,clay,phaq,sand,silt,orgc,slopemean,bedrock,cfr,temperature,landcover,precipitation,ecoregion
2,1149115,0_30,2004-5-27,-92.924872,15.375747,12.576397,5.882000,42.139379,45.284223,11.860177,0.153699,rocas_metamorficas,0.001939,16.675776,Tropical húmeda,147.141876,Terrestre
5,1149116,0_30,2004-11-12,-93.805531,15.933933,9.141012,6.183665,86.473648,4.385340,7.997192,0.153699,rocas_metamorficas,0.001939,14.863654,Inundable o Transición tierra-mar,85.933456,Terrestre
9,1149118,0_30,1984-4-15,-92.340858,14.761703,21.426717,6.282607,49.860114,28.713169,15.670057,0.153699,rocas_igneas,0.001939,15.884959,Tropical húmeda,77.329765,Terrestre
12,1149120,0_30,1990-7-12,-104.979876,19.941109,9.880670,4.935505,72.640192,17.479138,2.013396,2.462867,rocas_igneas,2.349696,15.541919,Templada subhúmeda,159.122009,Terrestre
15,1149123,0_30,1990-7-7,-104.616548,19.695687,32.477529,4.309409,50.851700,16.670771,9.258772,0.459251,rocas_sedimentarias,0.221472,15.541919,Tropical subhúmeda,159.122009,Terrestre


In [59]:
# Save the merged dataset
output_path = os.path.join(base_path, "datasets", "Mexico")
os.makedirs(output_path, exist_ok=True)
output_file = os.path.join(output_path, 'Mexico_single_depth_orgc_merged.csv')
merged_df.to_csv(output_file, index=False)
print(f"Merged file saved to: {output_file}")

Merged file saved to: d:\tierra\datasets\Mexico\Mexico_single_depth_orgc_merged.csv
